# Approximate Bayesian Computation

We compare the results with two algorithms in ABC, i.e. we compare the basic formulation of ABC with Importance Sampling ABC.

Let $x = (x_1, \cdots, x_{100})$ be an i.i.d from the following mixture model:

$$
\lambda \cdot Bin(x_i | \theta_1, N = 4) + (1 - \lambda) \cdot Bin(x_i| \theta_2, N=4)
$$

The parameters of interest are $\lambda$, $\theta_1$ and $\theta_2$. We want to get samples from
the posterior distribution

$$
p(\lambda, \theta_1, \theta_2 | x_1, \cdots, x_{100})
$$

by assuming the following prior distributions:

- $\lambda \sim U(0,1)$
- $(\theta_1, \theta_2)$ uniformly distributed over the set $\{(\theta_1, \theta_2): 0 \le \theta_2 \le \theta_1 \le 1\}$


## Initial Representation of the mixture

For this example we will use:

- $\theta_1 = 0.7$
- $\theta_2 = 0.2$
- $\lambda = 0.8$

Sampling from the real mixture $100$ observations we get the following histogram:


In [ ]:
import matplotlib.pyplot as plt
import torch
from simulator import GenerativeProcess, BinomialMixtureSimulator
import seaborn as sns

sns.set_theme(style="darkgrid")

torch.manual_seed(1)
x_obs = GenerativeProcess(0.7, 0.2, 0.8).mixture.sample((100,))
s_obs = x_obs.mean()
print(s_obs)
sns.histplot(x_obs, discrete=True)

## Basic ABC Algorithm

In the basic formulation of ABC:

1. Propose parameters
2. Use a simulator to generate samples with the proposed parameters
3. Compute a statistic and a distance between statistics of observed data and simulated data
4. If the distance is smaller than a threshold, accept the proposed parameters

In [ ]:
from sampler.base_abc import BaseABC

simulator = BinomialMixtureSimulator()

algo = BaseABC(
    observations=x_obs,
    simulator=simulator,
    summary_statistic="frequency",
    threshold=0.05,
)

n_sim = 1000
accepted = algo.compute(n_sim)
print(f"Acceptance ratio: {len(accepted) / n_sim * 100:.2f}%")

The basic algorithm is computationally intensive, and it rejects lots of proposals that do not match the similarity condition.
After having at least one triplet of parameters accepted, we can use them to simulate as many data as we want from the simulator.

In [ ]:
new_samples = simulator.generate(*accepted[0], times=1000)
sns.histplot(new_samples, discrete=True)
a_params = accepted[0]
plt.title(f"Theta1: {a_params[0]:.2f}, Theta2: {a_params[1]:.2f}, Rate: {a_params[2]:.2f}")
plt.show()

## Importance Sampling ABC

Instead of rejecting proposed parameters we can give a weight to all of them and calculate a weighted average to get the expectation of that parameter.

To do so we need to choose some proposals (apart from the priors) for the parameters of interests.
In our case we choose 2 Beta distributions.

In [ ]:
from importance_sampling import ImportanceSampling

imp_s = ImportanceSampling(
    observations=x_obs, simulator=simulator, summary_statistic="frequency", threshold=0.1
)

imp_s.plot_proposals()

The fraction between the density of the prior and the density of the proposal gives the unnormalized weight of a specific parameter.

Given a prior $p$ and a proposal $g$ for a parameter $\theta$, to get the unnormalized weight $\tilde{w}_i$ of a specific $\theta_i \sim p$, we calculate the following ratio:
$$
\frac{p(\theta_i)}{g(\theta_i)}
$$

To get the normalized weight $w_i$ we simply calculate all the $N$ unnormalized weights and the total sum is the normalizing constant

$$
w_i = \frac{\tilde{w}_i}{\sum^N_{i=1 } w_i}
$$

The parameters that would be discarded in the previous algorithm are now weighted and used to calculate the expectation.

Using Importance Sampling will lead to the following parameters:


In [ ]:
t1, t2, r = imp_s.compute()

print(f"Expected Theta_1: {t1}\n"
      f"Expected Theta_2: {t2}\n"
      f"Expected Rate: {r}\n")

Like in the basic algorithm we can use the parameters to sample from the simulator.

In [ ]:
is_s = simulator.generate(
    t1, t2, r, times=1000
)
sns.histplot(is_s, discrete=True)